# 02 — Deteksi Wajah & Cropping (InsightFace)

Notebook lanjutan `identity-lora-pipeline`: membaca hasil crop tubuh penuh dari
`data/cropped/<identitas>/` (modul 001), mendeteksi wajah dengan InsightFace
`buffalo_sc`, memilih bbox dengan confidence tertinggi jika ada beberapa wajah,
lalu memotong region wajah dengan padding ke `data/face/<identitas>/`.

Alur: `data/cropped/<identitas>/` → deteksi wajah (K8) → pilih confidence tertinggi
→ crop + padding (K14, K15) → folder output `data/face/<identitas>/`. Gambar tanpa
wajah dilewati.

Kelas error dimuat dari `00_pipeline_errors.ipynb`. Sel runner (tag `manual-run`)
memakai hasil crop sungguhan di `data/cropped/` dan hanya menampilkan `DISPLAY_LIMIT`
data pertama.


## 1. Import dan logger

In [ ]:
import logging
import sys
from dataclasses import dataclass, field
from enum import Enum
from itertools import islice
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from insightface.app import FaceAnalysis

# Contoh: 2026-09-26 10:30:00,123 | INFO     | face_detection_and_cropping | ...
LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"


def setup_logger(name: str) -> logging.Logger:
    """Membuat logger dengan format teks.

    Args:
        name: Nama logger.

    Returns:
        Logger yang siap dipakai.
    """
    created = logging.getLogger(name)
    if created.handlers:
        return created

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(LOG_FORMAT))
    created.addHandler(handler)
    created.setLevel(logging.INFO)
    return created


logger = setup_logger("face_detection_and_cropping")

## 2. Konfigurasi

In [ ]:
# Kernel notebook biasanya berjalan dari folder notebooks/, jadi path relatif "data/raw"
# tidak ketemu. Root project dicari dengan naik dari folder kerja sampai ada data/raw/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "raw").is_dir()),
    Path.cwd(),
)
CROPPED_DIR = PROJECT_ROOT / "data" / "cropped"  # input, hasil modul 001 (K7)
FACE_DIR = PROJECT_ROOT / "data" / "face"  # output crop wajah (K13)

FACE_MODEL_NAME = "buffalo_sc"
FACE_DET_THRESH = 0.5  # sementara, belum dikalibrasi pada data asli (K8)

PADDING_RATIO = 0.7  # padding simetris di semua sisi bounding box sebelum crop (K14)
MIN_SIDE_PX = 64  # sisi terpanjang minimum hasil crop, sementara (K14)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

DISPLAY_LIMIT = 5  # jumlah data pertama yang ditampilkan sel runner

## 3. Error pipeline

In [ ]:
# Definisi error ada di notebook terpisah; %run mengeksekusinya di namespace kernel ini.
%run "{PROJECT_ROOT / 'notebooks' / '00_pipeline_errors.ipynb'}"

## 4. Deteksi wajah (K8)

In [ ]:
@dataclass
class BoundingBox:
    """Satu box hasil deteksi wajah."""

    x1: int
    y1: int
    x2: int
    y2: int
    confidence: float


def load_face_analyzer(model_name: str, det_thresh: float) -> FaceAnalysis:
    """Memuat model deteksi wajah InsightFace.

    Args:
        model_name: Nama model pack InsightFace, mis. "buffalo_sc".
        det_thresh: Ambang confidence minimum deteksi wajah.

    Returns:
        FaceAnalysis yang siap dipakai `detect_faces`, hanya memuat modul deteksi.
    """
    analyzer = FaceAnalysis(name=model_name, allowed_modules=["detection"])
    analyzer.prepare(ctx_id=-1, det_thresh=det_thresh)
    return analyzer


def detect_faces(analyzer: FaceAnalysis, image: np.ndarray) -> list[BoundingBox]:
    """Mendeteksi wajah pada satu gambar.

    Args:
        analyzer: FaceAnalysis hasil `load_face_analyzer`.
        image: Gambar hasil `cv2.imread`.

    Returns:
        Daftar box wajah beserta confidence-nya.

    Raises:
        FaceDetectionError: Kalau model gagal melakukan inferensi pada gambar.
    """
    try:
        faces = analyzer.get(image)
    except (RuntimeError, ValueError, cv2.error) as e:
        logger.error(f"Deteksi wajah gagal: {e}", exc_info=True)
        raise FaceDetectionError(f"Deteksi wajah gagal: {e}") from e
    return [
        BoundingBox(
            x1=int(face.bbox[0]),
            y1=int(face.bbox[1]),
            x2=int(face.bbox[2]),
            y2=int(face.bbox[3]),
            confidence=float(face.det_score),
        )
        for face in faces
    ]


def validate_single_face(boxes: list[BoundingBox]) -> BoundingBox:
    """Memilih bbox wajah dengan confidence tertinggi pada satu gambar.

    Args:
        boxes: Daftar box hasil `detect_faces`.

    Returns:
        Box wajah dengan confidence tertinggi.

    Raises:
        NoFaceDetectedError: Kalau tidak ada wajah sama sekali.
    """
    if len(boxes) == 0:
        raise NoFaceDetectedError("Tidak ada wajah terdeteksi")
    return max(boxes, key=lambda box: box.confidence)

In [ ]:
# Runner "define -> demonstrate": menjalankan detect_faces pada DISPLAY_LIMIT gambar
# pertama di data/cropped/ (hasil modul 001) dan menggambar box wajah dengan confidence
# tertinggi. Butuh data/cropped/<identitas>/ terisi (jalankan notebook 01 lebih dulu)
# dan koneksi internet (untuk mengunduh bobot buffalo_sc saat pertama kali dijalankan)
# — sengaja tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md); Arya
# yang menjalankannya setelah mengisi data.
_runner_image_paths = list(
    islice(
        (
            image_path
            for identity_dir in sorted(CROPPED_DIR.iterdir())
            if identity_dir.is_dir()
            for image_path in sorted(identity_dir.iterdir())
            if image_path.suffix.lower() in IMAGE_EXTENSIONS
        ),
        DISPLAY_LIMIT,
    )
)

_runner_analyzer = load_face_analyzer(FACE_MODEL_NAME, FACE_DET_THRESH)
_runner_images = [cv2.imread(str(path)) for path in _runner_image_paths]
_runner_faces = [detect_faces(_runner_analyzer, image) for image in _runner_images]

_runner_figure, _runner_axes = plt.subplots(
    1, len(_runner_image_paths), figsize=(4 * len(_runner_image_paths), 6), squeeze=False
)
for _runner_axis, _runner_path, _runner_image, _runner_boxes in zip(
    _runner_axes[0], _runner_image_paths, _runner_images, _runner_faces
):
    _runner_display = _runner_image.copy()
    try:
        _runner_box = validate_single_face(_runner_boxes)
    except NoFaceDetectedError as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        _runner_axis.imshow(cv2.cvtColor(_runner_display, cv2.COLOR_BGR2RGB))
        _runner_axis.set_title(_runner_path.name, fontsize=8)
        _runner_axis.axis("off")
        continue
    cv2.rectangle(
        _runner_display,
        (_runner_box.x1, _runner_box.y1),
        (_runner_box.x2, _runner_box.y2),
        (0, 255, 0),
        2,
    )
    cv2.putText(
        _runner_display,
        f"{_runner_box.confidence:.2f}",
        (_runner_box.x1, max(_runner_box.y1 - 10, 0)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2,
    )
    _runner_axis.imshow(cv2.cvtColor(_runner_display, cv2.COLOR_BGR2RGB))
    _runner_axis.set_title(_runner_path.name, fontsize=8)
    _runner_axis.axis("off")
plt.show()

for _runner_path, _runner_boxes in zip(_runner_image_paths, _runner_faces):
    try:
        _runner_box = validate_single_face(_runner_boxes)
    except NoFaceDetectedError as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        continue
    print(
        f"{_runner_path.name}: bbox terpilih "
        f"({(_runner_box.x1, _runner_box.y1, _runner_box.x2, _runner_box.y2, _runner_box.confidence)})"
    )

## 5. Cropping wajah (K14, K15)

In [ ]:
def _apply_padding(
    box: BoundingBox, image_shape: tuple[int, int], padding_ratio: float
) -> tuple[int, int, int, int]:
    height, width = image_shape
    box_width = box.x2 - box.x1
    box_height = box.y2 - box.y1
    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)

    x1 = max(box.x1 - pad_x, 0)
    y1 = max(box.y1 - pad_y, 0)
    x2 = min(box.x2 + pad_x, width)
    y2 = min(box.y2 + pad_y, height)
    return x1, y1, x2, y2


def _is_large_enough(region: np.ndarray, min_side_px: int) -> bool:
    height, width = region.shape[:2]
    if height == 0 or width == 0:
        return False
    return max(height, width) >= min_side_px


def crop_with_padding(
    image: np.ndarray, box: BoundingBox, padding_ratio: float, min_side_px: int
) -> np.ndarray:
    """Memotong region wajah dengan padding, menolak hasil yang terlalu kecil.

    Args:
        image: Gambar asal, hasil `cv2.imread`.
        box: Box wajah hasil `validate_single_face`.
        padding_ratio: Rasio padding simetris di semua sisi box.
        min_side_px: Sisi terpanjang minimum hasil crop yang masih diterima.

    Returns:
        Region gambar hasil crop.

    Raises:
        CropTooSmallError: Kalau hasil crop kosong atau terlalu kecil.
    """
    x1, y1, x2, y2 = _apply_padding(box, image.shape[:2], padding_ratio)
    region = image[y1:y2, x1:x2]
    if not _is_large_enough(region, min_side_px):
        raise CropTooSmallError(
            f"Crop {region.shape[1]}x{region.shape[0]}px kosong atau di bawah ambang {min_side_px}px"
        )
    return region


def save_crop(region: np.ndarray, output_path: Path) -> None:
    """Menyimpan hasil crop ke berkas."""
    try:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        ok = cv2.imwrite(str(output_path), region)
    except OSError as e:
        logger.error(f"Crop gagal disimpan ke {output_path}: {e}", exc_info=True)
        raise CropSaveError(f"Crop gagal disimpan ke {output_path}: {e}") from e
    if not ok:
        raise CropSaveError(f"Crop gagal disimpan ke {output_path}")


def build_crop_filename(source_path: Path, suffix: str = "face") -> str:
    """Membentuk nama file hasil crop dari nama file asal."""
    return f"{source_path.stem}_{suffix}.jpg"

In [ ]:
# Runner "define -> demonstrate": menerapkan crop_with_padding pada bbox wajah dengan
# confidence tertinggi dari runner deteksi sebelumnya, lalu menampilkan hasil crop-nya.
# Sengaja tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md).
_runner_face_crops = []
for _runner_path, _runner_image, _runner_boxes in zip(
    _runner_image_paths, _runner_images, _runner_faces
):
    try:
        _runner_box = validate_single_face(_runner_boxes)
    except NoFaceDetectedError as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        continue
    try:
        _runner_region = crop_with_padding(_runner_image, _runner_box, PADDING_RATIO, MIN_SIDE_PX)
    except CropTooSmallError as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        continue
    print(
        f"{build_crop_filename(_runner_path)}: "
        f"{_runner_region.shape[1]}x{_runner_region.shape[0]}px"
    )
    _runner_face_crops.append(_runner_region)

if _runner_face_crops:
    _runner_figure, _runner_axes = plt.subplots(
        1, len(_runner_face_crops), figsize=(4 * len(_runner_face_crops), 6), squeeze=False
    )
    for _runner_axis, _runner_region in zip(_runner_axes[0], _runner_face_crops):
        _runner_axis.imshow(cv2.cvtColor(_runner_region, cv2.COLOR_BGR2RGB))
        _runner_axis.axis("off")
    plt.show()

## 6. Orkestrasi deteksi dan cropping wajah

In [ ]:
class RegionOutcome(Enum):
    """Kategori hasil pemrosesan crop wajah."""

    CROPPED = "cropped"
    SKIPPED_NO_DETECTION = "skipped_no_detection"
    SKIPPED_TOO_SMALL = "skipped_too_small"
    SKIPPED_UNREADABLE = "skipped_unreadable"


@dataclass
class RegionSummary:
    """Ringkasan hasil pemrosesan crop wajah."""

    cropped: int = 0
    skipped_no_detection: int = 0
    skipped_too_small: int = 0
    skipped_unreadable: int = 0

    def add(self, outcome: RegionOutcome) -> None:
        """Menambah satu hitungan sesuai kategori `outcome`."""
        setattr(self, outcome.value, getattr(self, outcome.value) + 1)


def total_skipped(summary: RegionSummary) -> int:
    """Menghitung total gambar yang dilewati."""
    return (
        summary.skipped_no_detection
        + summary.skipped_too_small
        + summary.skipped_unreadable
    )


@dataclass
class ProcessSummary:
    """Ringkasan hasil pemrosesan satu identitas."""

    face: RegionSummary = field(default_factory=RegionSummary)


def _read_image(image_path: Path) -> np.ndarray:
    try:
        image = cv2.imread(str(image_path))
    except cv2.error as e:
        logger.error(f"Gambar tidak bisa dibaca: {image_path}: {e}", exc_info=True)
        raise ImageReadError(f"Gambar tidak bisa dibaca: {image_path}") from e
    if image is None:
        raise ImageReadError(f"Gambar tidak bisa dibaca (format tidak dikenali): {image_path}")
    return image


def _list_images(identity_dir: Path) -> list[Path]:
    return sorted(p for p in identity_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def _crop_and_save(
    image: np.ndarray,
    box: BoundingBox,
    output_path: Path,
    padding_ratio: float,
    min_side_px: int,
) -> RegionOutcome:
    try:
        region = crop_with_padding(image, box, padding_ratio, min_side_px)
    except CropTooSmallError as e:
        logger.warning(f"Dilewati, {e}: {output_path}")
        return RegionOutcome.SKIPPED_TOO_SMALL
    save_crop(region, output_path)
    return RegionOutcome.CROPPED


def _process_face(
    image: np.ndarray,
    source_path: Path,
    output_dir: Path,
    face_analyzer: FaceAnalysis,
    padding_ratio: float,
    min_side_px: int,
) -> RegionOutcome:
    boxes = detect_faces(face_analyzer, image)
    try:
        box = validate_single_face(boxes)
    except NoFaceDetectedError:
        logger.info(f"Wajah dilewati, tidak ada wajah terdeteksi: {source_path}")
        return RegionOutcome.SKIPPED_NO_DETECTION
    output_path = output_dir / build_crop_filename(source_path)
    return _crop_and_save(image, box, output_path, padding_ratio, min_side_px)


def process_image(
    image_path: Path,
    face_dir: Path,
    face_analyzer: FaceAnalysis,
    padding_ratio: float,
    min_side_px: int,
) -> RegionOutcome:
    """Memproses satu gambar untuk crop wajah."""
    try:
        image = _read_image(image_path)
    except ImageReadError as e:
        logger.warning(str(e))
        return RegionOutcome.SKIPPED_UNREADABLE
    return _process_face(image, image_path, face_dir, face_analyzer, padding_ratio, min_side_px)


def process_identity(
    identity_dir: Path,
    face_dir: Path,
    face_analyzer: FaceAnalysis,
    padding_ratio: float,
    min_side_px: int,
) -> ProcessSummary:
    """Menjalankan deteksi dan cropping wajah untuk satu identitas."""
    if not identity_dir.exists():
        logger.warning(f"Folder identitas tidak ditemukan: {identity_dir}")
        return ProcessSummary()

    face_dir.mkdir(parents=True, exist_ok=True)
    summary = ProcessSummary()
    for image_path in _list_images(identity_dir):
        outcome = process_image(image_path, face_dir, face_analyzer, padding_ratio, min_side_px)
        summary.face.add(outcome)

    logger.info(f"[{identity_dir.name}] wajah: {summary.face.cropped} crop")
    return summary


def run_pipeline(identities: dict[str, Path]) -> dict[str, ProcessSummary]:
    """Menjalankan pipeline deteksi dan cropping wajah untuk setiap identitas."""
    face_analyzer = load_face_analyzer(FACE_MODEL_NAME, FACE_DET_THRESH)

    summaries: dict[str, ProcessSummary] = {}
    for identity_name, identity_dir in identities.items():
        summaries[identity_name] = process_identity(
            identity_dir,
            FACE_DIR / identity_name,
            face_analyzer,
            PADDING_RATIO,
            MIN_SIDE_PX,
        )
    return summaries

In [ ]:
# Runner "define -> demonstrate": menjalankan pipeline untuk semua folder di
# data/cropped/ (hasil modul 001) dan menampilkan ringkasan DISPLAY_LIMIT identitas
# pertama. Jalankan notebook 01 sampai selesai lebih dulu supaya data/cropped/
# terisi. Sel ini memanggil model InsightFace sungguhan dan mengunduh bobot
# buffalo_sc saat pertama kali dijalankan — sengaja tidak dijalankan otomatis saat
# pemeriksaan kode (lihat README.md).
IDENTITIES = {
    identity_dir.name: identity_dir
    for identity_dir in sorted(CROPPED_DIR.iterdir())
    if identity_dir.is_dir()
} if CROPPED_DIR.exists() else {}

summaries = run_pipeline(IDENTITIES)
for identity_name, summary in islice(summaries.items(), DISPLAY_LIMIT):
    print(
        f"{identity_name}: wajah {summary.face.cropped} crop/"
        f"{total_skipped(summary.face)} dilewati"
    )